# Wheat threshold 3: paired capture and audit

Run this notebook in the existing live Kaggle session containing the checkpoints. A new session does not inherit `/kaggle/working` files. Enable Internet and allow the notebook access to the Kaggle Secret `GITHUB_TOKEN`.

This is a runnable handoff only. It runs four candidate arms, two seeds, both seats: 16 official-engine games. Real-checkpoint gameplay conclusions remain pending until the user runs it.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, signal, subprocess, sys, tempfile, time, zipfile

ROOT = Path('/kaggle/working/interactive_curriculum_0acfe858')
P_FINAL = ROOT / 'runs/P_bce_fullspace_lr3e5_from_Ou10_s43049/final.npz'
BC_E = Path('/kaggle/input/datasets/billll/v0-bc-e/best.pt')
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
BRANCH = 'codex/stage25-upkeep-ablation'
CODE_SHA = 'SET_TO_RELEASE_COMMIT'
SEEDS = [144368101, 2112243121]
MASTER_SEED = 25
VARIANTS = ['fertilizer', 'fertilizer_wheat3', 'combined', 'combined_wheat3']
JAX_PLATFORM = 'cpu'

for path in (ROOT, P_FINAL, BC_E):
    if not path.exists():
        raise FileNotFoundError(path)
RUN_ROOT = ROOT / ('wheat_threshold_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
RUN_ROOT.mkdir(exist_ok=False)
REPO = RUN_ROOT / 'repo'
OUTPUT = RUN_ROOT / 'results'
CAPTURES = RUN_ROOT / 'captures'
AUDIT = RUN_ROOT / 'audit'
print('planned games:', len(SEEDS) * 2 * len(VARIANTS))
print('run root:', RUN_ROOT)

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret(GITHUB_SECRET_NAME)
if not token:
    raise RuntimeError('GITHUB_TOKEN secret is empty')
try:
    with tempfile.TemporaryDirectory(prefix='wheat_git_') as tmp:
        askpass = Path(tmp) / 'askpass.py'
        askpass.write_text('#!' + sys.executable + '\n' +
                          'import os,sys\n' +
                          'print("x-access-token" if "username" in sys.argv[1].lower() else os.environ["WHEAT_GIT_TOKEN"])\n')
        askpass.chmod(0o700)
        env = {**os.environ, 'GIT_ASKPASS': str(askpass),
               'GIT_TERMINAL_PROMPT': '0', 'WHEAT_GIT_TOKEN': token}
        subprocess.run(['git', 'clone', '--depth', '10', '--single-branch',
                        '--branch', BRANCH,
                        'https://github.com/BillXu21/Kaggriculture.git', str(REPO)],
                       env=env, check=True, timeout=180)
        subprocess.run(['git', 'checkout', '--detach', CODE_SHA], cwd=REPO,
                       env=env, check=True, timeout=30)
finally:
    token = None
    if 'env' in globals():
        env.pop('WHEAT_GIT_TOKEN', None)
actual_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert actual_sha == CODE_SHA
print('pinned source:', actual_sha)

In [ ]:
EVAL_ENV = os.environ.copy()
EVAL_ENV.update({
    'PYTHONUNBUFFERED': '1', 'OMP_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1', 'OPENBLAS_NUM_THREADS': '1',
    'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
    'PYTHONPATH': str(REPO),
})
if JAX_PLATFORM:
    EVAL_ENV['JAX_PLATFORMS'] = JAX_PLATFORM
guard = 'from oracle.provenance import require_official_modules; require_official_modules(); print("official engine provenance passed")'
subprocess.run([sys.executable, '-c', guard], cwd=REPO, env=EVAL_ENV, check=True)
preflight = r'''
import sys, jax, torch, optax, pyarrow
from bc_manager_jax.checkpoint import load_torch_checkpoint
from bc_manager_jax.model import ManagerConfig
from rl_manager.ppo_checkpoint import load_ppo_checkpoint
params, metadata = load_torch_checkpoint(sys.argv[2], expected_e_history_version='E_LEGACY')
config = ManagerConfig(**metadata['model_config'])
state, meta = load_ppo_checkpoint(sys.argv[1], config=config, expected_e_history_version='E_LEGACY')
print({'jax': jax.__version__, 'devices': [str(d) for d in jax.devices()], 'history': meta['e_history_version']})
'''
subprocess.run([sys.executable, '-c', preflight, str(P_FINAL), str(BC_E)],
               cwd=REPO, env=EVAL_ENV, check=True)

## Exact-identity capture smoke

Run one candidate game with and without capture. Stop if the bank, opponent bank, status, or trace identity differs.

In [ ]:
def run_eval(extra, output_dir, log_path):
    command = [sys.executable, '-m', 'tools.evaluate_stage25_upkeep',
               '--checkpoint', str(P_FINAL), '--e-checkpoint', str(BC_E),
               '--backend', 'official', '--e-history-version', 'E_LEGACY',
               '--master-seed', str(MASTER_SEED), '--seeds', *map(str, SEEDS),
               '--variants', 'fertilizer', '--game-filter', f'{SEEDS[0]}:0',
               '--output-dir', str(output_dir), *extra]
    with log_path.open('w') as log:
        process = subprocess.Popen(command, cwd=REPO, env=EVAL_ENV, text=True,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in process.stdout:
            log.write(line); log.flush(); print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError('identity smoke failed')

smoke_plain = RUN_ROOT / 'smoke_plain'
smoke_cap = RUN_ROOT / 'smoke_cap'
smoke_captures = RUN_ROOT / 'smoke_captures'
run_eval([], smoke_plain, RUN_ROOT / 'smoke_plain.log')
run_eval(['--capture-dir', str(smoke_captures)], smoke_cap, RUN_ROOT / 'smoke_cap.log')
print('Smoke complete. Inspect both JSON outputs before continuing if needed.')

In [ ]:
command = [sys.executable, '-m', 'tools.evaluate_stage25_upkeep',
           '--checkpoint', str(P_FINAL), '--e-checkpoint', str(BC_E),
           '--output-dir', str(OUTPUT), '--capture-dir', str(CAPTURES),
           '--backend', 'official', '--e-history-version', 'E_LEGACY',
           '--master-seed', str(MASTER_SEED), '--seeds', *map(str, SEEDS),
           '--variants', *VARIANTS]
(RUN_ROOT / 'launch.json').write_text(json.dumps({'command': command,
    'source_commit': CODE_SHA, 'seeds': SEEDS, 'master_seed': MASTER_SEED,
    'variants': VARIANTS, 'engine': 'official kaggle_environments 1.32.7',
    'candidate_checkpoint': str(P_FINAL), 'bc_e_checkpoint': str(BC_E)}, indent=2))
with (RUN_ROOT / 'run.log').open('w') as log:
    process = subprocess.Popen(command, cwd=REPO, env=EVAL_ENV, text=True,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in process.stdout:
        log.write(line); log.flush(); print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError('paired evaluation failed; partial results remain')
print('completed 16 games')

In [ ]:
audit_command = [sys.executable, '-m', 'tools.audit_stage25_capture',
                 '--capture-dir', str(CAPTURES), '--output-dir', str(AUDIT),
                 '--variants', *VARIANTS, '--focus-pairs', '8']
audit = subprocess.run(audit_command, cwd=REPO, env=EVAL_ENV, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(RUN_ROOT / 'audit.log').write_text(audit.stdout)
print(audit.stdout)
if audit.returncode:
    raise RuntimeError('audit failed')
print((AUDIT / 'audit.md').read_text()[:12000])

In [ ]:
provenance = {
    'source_commit': CODE_SHA,
    'branch': BRANCH,
    'candidate_checkpoint': str(P_FINAL),
    'bc_e_checkpoint': str(BC_E),
    'candidate_sha256': hashlib.sha256(P_FINAL.read_bytes()).hexdigest(),
    'bc_e_sha256': hashlib.sha256(BC_E.read_bytes()).hexdigest(),
    'seeds': SEEDS, 'master_seed': MASTER_SEED, 'variants': VARIANTS,
    'engine': 'official kaggle_environments 1.32.7',
    'note': 'No real-checkpoint conclusion is made by repository-local validation.'
}
(RUN_ROOT / 'provenance.json').write_text(json.dumps(provenance, indent=2) + '\n')
zip_path = ROOT / ('wheat_threshold_capture_' + RUN_ROOT.name.split('_', 1)[-1] + '.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in RUN_ROOT.rglob('*'):
        if path.is_file() and path != zip_path:
            archive.write(path, path.relative_to(RUN_ROOT))
print('downloadable ZIP:', zip_path)